In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

from scipy.stats import skew, kurtosis

from skimage.measure import shannon_entropy

from src.utils import load_image

# ============================================================
# CONFIG
# ============================================================

CSV_INPUT = "stenosis_windows_dataset.csv"

CSV_OUTPUT = "stenosis_windows_gabor_features.csv"

# ============================================================
# CARGAR CSV
# ============================================================

df = pd.read_csv(CSV_INPUT)

print(f"Total windows loaded: {len(df)}")

# ============================================================
# PARÁMETROS GABOR
# ============================================================

ksize = 21

sigmas = [1, 3]

thetas = [
    0,
    np.pi / 4,
    np.pi / 2,
    3 * np.pi / 4
]

lambdas = [5, 10]

gammas = [0.5, 1.0]

# ============================================================
# CREAR BANCO DE FILTROS
# ============================================================

gabor_filters = []

filter_names = []

for sigma in sigmas:

    for theta in thetas:

        for lambd in lambdas:

            for gamma in gammas:

                kernel = cv2.getGaborKernel(

                    (ksize, ksize),
                    sigma,
                    theta,
                    lambd,
                    gamma,
                    psi=0,
                    ktype=cv2.CV_32F
                )

                gabor_filters.append(kernel)

                filter_name = (
                    f"s{sigma}_"
                    f"t{round(theta,2)}_"
                    f"l{lambd}_"
                    f"g{gamma}"
                )

                filter_names.append(filter_name)

print(f"Total Gabor filters: {len(gabor_filters)}")

# ============================================================
# EXTRAER FEATURES
# ============================================================

feature_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    # --------------------------------------------------------
    # CARGAR IMAGEN
    # --------------------------------------------------------

    img = load_image(row['image_path'])

    if img is None:
        continue

    # --------------------------------------------------------
    # EXTRAER ROI
    # --------------------------------------------------------

    xmin = int(row['xmin'])
    ymin = int(row['ymin'])
    xmax = int(row['xmax'])
    ymax = int(row['ymax'])

    roi = img[ymin:ymax, xmin:xmax]

    if roi.size == 0:
        continue

    # ========================================================
    # FEATURES BASE
    # ========================================================

    feature_dict = {

        'window_id': row['window_id'],

        'label': row['label']
    }

    # ========================================================
    # FEATURES GABOR
    # ========================================================

    for kernel, fname in zip(gabor_filters, filter_names):

        filtered = cv2.filter2D(

            roi,
            cv2.CV_32F,
            kernel
        )

        vals = filtered.flatten()

        # ----------------------------------------------------
        # FEATURES IMPORTANTES
        # ----------------------------------------------------

        mean_val = np.mean(vals)

        std_val = np.std(vals)

        energy_val = np.sum(vals ** 2)

        entropy_val = shannon_entropy(vals)

        skewness_val = skew(vals)

        kurtosis_val = kurtosis(vals)

        # ----------------------------------------------------
        # GUARDAR
        # ----------------------------------------------------

        feature_dict[f"{fname}_mean"] = mean_val

        feature_dict[f"{fname}_std"] = std_val

        feature_dict[f"{fname}_energy"] = energy_val

        feature_dict[f"{fname}_entropy"] = entropy_val

        feature_dict[f"{fname}_skew"] = skewness_val

        feature_dict[f"{fname}_kurtosis"] = kurtosis_val

    # ========================================================
    # GUARDAR FILA
    # ========================================================

    feature_rows.append(feature_dict)

# ============================================================
# CREAR DATAFRAME
# ============================================================

features_df = pd.DataFrame(feature_rows)

print(features_df.head())

print(f"\nTotal feature rows: {len(features_df)}")

print(f"Total features: {features_df.shape[1]}")

# ============================================================
# GUARDAR CSV
# ============================================================

features_df.to_csv(

    CSV_OUTPUT,
    index=False
)

print(f"\nFeatures CSV guardado en: {CSV_OUTPUT}")

Total windows loaded: 5000
Total Gabor filters: 32


100%|██████████| 5000/5000 [08:15<00:00, 10.08it/s]


         window_id  label  s1_t0_l5_g0.5_mean  s1_t0_l5_g0.5_std  \
0  14_023_2_0093_1      0          742.579712         294.250122   
1  14_023_2_0093_2      0          770.617676         282.823517   
2  14_023_2_0093_3      0          756.729553         297.860748   
3  14_023_2_0093_4      0          728.272217         286.572968   
4  14_023_2_0093_5      0          759.083923         220.569870   

   s1_t0_l5_g0.5_energy  s1_t0_l5_g0.5_entropy  s1_t0_l5_g0.5_skew  \
0          4.083249e+09              12.643544           -0.072581   
1          4.312581e+09              12.643231           -0.275491   
2          4.232708e+09              12.643231           -0.360157   
3          3.920028e+09              12.643544            0.031978   
4          3.999101e+09              12.643231           -0.222613   

   s1_t0_l5_g0.5_kurtosis  s1_t0_l5_g1.0_mean  s1_t0_l5_g1.0_std  ...  \
0               -1.108089          371.246338         156.366638  ...   
1               -1.02309